In [ ]:
# %load_ext nb_mypy
# %nb_mypy Off

In [ ]:
from __future__ import annotations
import numpy as np
import random
import copy
import importlib
import matplotlib.pyplot as plt

from typing import Tuple, List
from dataclasses import replace
from numpy import array, zeros

# from Big_Class import Big_Class  # already imported one NETfuncs is imported
from User_Variables import User_Variables  # already imported one NETfuncs is imported
from Network_Structure import Network_Structure  # already imported one NETfuncs is imported
from Big_Class import Big_Class
from Network_State import Network_State
from Networkx_Net import Networkx_Net
import matrix_functions, functions, statistics, plot_funcs, solve, plot_funcs, colors

## colors

In [ ]:
importlib.reload(colors)

colors_lst, red, cmap = colors.color_scheme()
cmap

# Set up Network

In [ ]:
import config
importlib.reload(config)
CFG = config.CFG

In [ ]:
print('Configured Nout:', CFG.Strctr.Nout)

In [ ]:
import Network_Structure
importlib.reload(Network_Structure)
from Network_Structure import Network_Structure
importlib.reload(matrix_functions)

## Structure is the first simulation class instance
Strctr = Network_Structure(CFG)
Strctr.build_incidence(Strctr.net_type)
BigClass = Big_Class(Strctr)

print('input_nodes_arr ', Strctr.input_nodes_arr)
print('extraInput_nodes_arr ', Strctr.extraInput_nodes_arr)
print('inter_nodes_arr ', Strctr.inter_nodes_arr)
print('output_nodes_arr ', Strctr.output_nodes_arr)
print('extraOutput_nodes_arr ', Strctr.extraOutput_nodes_arr)
print('ground_nodes_arr ', Strctr.ground_nodes_arr)

import User_Variables, Supervisor_Class
importlib.reload(User_Variables)
importlib.reload(Supervisor_Class)
from User_Variables import User_Variables
from Supervisor_Class import Supervisor

Variabs = User_Variables(CFG)
Sprvsr = Supervisor(CFG, Strctr, Variabs)
BigClass.add_Variabs(Variabs)
BigClass.add_Sprvsr(Sprvsr)

In [ ]:
if Sprvsr.training_scheme in ['GD_like', 'Adaline']:
    Strctr.build_inverse_incidence()
    if Sprvsr.training_scheme == 'GD_like':
        Strctr.build_RM()

# build fictitious Network_Structure as if no inter nodes
if Sprvsr.training_scheme == 'Adaline' and Strctr.Ninter > 0:
    fict_cfg = replace(CFG, Strctr=replace(CFG.Strctr, Ninter=0))
    Strctr_fict = Network_Structure(fict_cfg)
    Strctr_fict.build_incidence(Strctr_fict.net_type)
    Strctr_fict.build_inverse_incidence()
    BigClass.add_Strctr_fict(Strctr_fict)  # add to big class

In [ ]:
# print('U', Strctr.DM)
# print('U_dagger', Strctr.DM_dagger)
# print('grad_loss_vec', State.grad_loss_vec)

In [ ]:
import Network_State
importlib.reload(Network_State)
from Network_State import Network_State

importlib.reload(plot_funcs)

## Initiate internal flow network state class
State = Network_State(CFG, BigClass)
if Sprvsr.task_type == 'Iris_classification':
    State.initiate_resistances(BigClass, CFG.State.R_vec_i)
    State.initiate_accuracy_vec(BigClass)
else:
#     R_mat_i = np.concatenate((1/Sprvsr.M, np.ones([1, 5])), axis=0)
#     R_vec_i = R_mat_i.flatten()
    # State.initiate_resistances(BigClass, np.matmul(Strctr.DM, np.matmul(Strctr.DM_dagger, R_vec_i)))
    State.initiate_resistances(BigClass, CFG.State.R_vec_i)
    print(State.R_in_t[0])
BigClass.add_State(State)  # add to big class

import Networkx_Net
importlib.reload(Networkx_Net)
from Networkx_Net import Networkx_Net

## build network graphics class and plot structure
NET = Networkx_Net(CFG)
NET.buildNetwork(BigClass)
NET.build_pos_lattice(BigClass, plot=True, node_labels=False)
BigClass.add_NET(NET)  # add to big class

# Train

In [ ]:
GD_cost_vec = []
dK_arr = []
dK_GD_arr = []
cosine_vec = []
cosine_lin = []
cosine_nonlin = []
cosine_optimal = []
dK_step=10**(-8)

def training_loop():
    for i in range(Sprvsr.iterations):

        # if task is classification and iteration # is beginning of epoch
        # draw output of network as output of mean of Irises
        if i % np.shape(Sprvsr.X_train)[0] == 0 and Sprvsr.task_type == 'Iris_classification':
            State.assign_targets_Iris(BigClass)
            # print('targets_mat', State.targets_mat)

        # staying stay_sample iteration under same sample
        if Sprvsr.use_p_tag and Sprvsr.noise_to_extra:
            k = 2*(i//Sprvsr.stay_sample) + 1
            if not(i%4):
                k-=1
        elif Sprvsr.use_p_tag and not(Sprvsr.noise_to_extra):
            k = (i//Sprvsr.stay_sample)*2 + i%2
        elif not(Sprvsr.use_p_tag) and Sprvsr.noise_to_extra:
            k = (i//Sprvsr.stay_sample)
        else:
            k = (i//Sprvsr.stay_sample)
        # print('k', k)

        for l in range(1):  # Repeat the block l times
            # # measurement modality
            # draw input and desired outputs from dataset and solve flow
            if not((i+1) % 4):  # add noise only at i=3 etc.
                State.draw_p_in_and_desired(Sprvsr, k, noise_to_extra=Sprvsr.noise_to_extra)
                State.solve_flow_given_modality(BigClass, "measure", noise_to_extra=Sprvsr.noise_to_extra)
            else:  # dont add noise to extra nodes
                State.draw_p_in_and_desired(Sprvsr, k)
                State.solve_flow_given_modality(BigClass, "measure")
    #         State.input_drawn_in_t.append(array([1]))
    #         State.desired_in_t.append(array([0.9, 0.1]))
    #         State.solve_flow_given_modality(BigClass, "measure")

            if Sprvsr.calculate_cosine_sim or Variabs.R_update == 'grad_desc':
                # GD cost
                # State.calc_GD_cost(Strctr, State.desired, mod='measure', func='mean_abs')
                State.calc_GD_cost(Strctr, State.desired, mod='measure', func='MSE')
                GD_cost_vec.append(State.GD_cost)

                # change in K grad desc
                # delta_K_GD = State.dK_grad_desc(Strctr, dK_step, State.desired, func='mean_abs')
                delta_K_GD = State.dK_grad_desc(Strctr, dK_step, State.desired, func='MSE')
                # print('delta_K_GD', delta_K_GD)    
                dK_GD_arr.append(delta_K_GD)

            # save R, p, u for network plots, after measurement
            if i == Sprvsr.iterations-1:
                NET.save_R_reordered(State.R_in_t[-1], Strctr.EIEJ_plots)
                NET.save_p_reordered(State.p)
                NET.save_u_reordered(State.u, Strctr.EIEJ_plots)
#                 plot_funcs.plotNetStructure(NET.NET, BigClass, NET.pos_lattice, node_labels=False, 
#                                     R_reordered=NET.R_reordered, u_reordered=NET.u_reordered, p_reordered=NET.p_reordered)

            if Sprvsr.include_Power:
                State.calc_Power_norm(BigClass)

            if not i % 2 and Sprvsr.use_p_tag:  # even iterations, take another sampled pressure and measure again
                pass
            else:  # odd iterations, go to update modality and update resistances
                if Strctr.net_type == 'beads':  # if beads - need a few iterations to converge
                    update_iters = 6
                else:  # if not beads, converges within a single iteration
                    update_iters = 1

                State.t += 1
                if not(State.t%Sprvsr.print_every):  # print time and outputs every print_every
                    # print('time=', State.t)
#                     print('measured output ', State.output)
#                     print('desired output ', State.desired)
                    # print('')
                    pass
                State.calc_loss(BigClass)
                # print('loss ', State.loss)
                loss_mean = np.mean(np.abs(State.loss), axis=1)

                if not((i + 1) % 4) and Sprvsr.access_interNodes:
                    State.update_inter(BigClass)
                else:
                    if Sprvsr.training_scheme in ['GD_like', 'Adaline']:
                        State.calc_update_vals_vec(BigClass)
                    State.update_input(BigClass)
                    State.update_output(BigClass)
#                     print('input_update_nxt ', State.input_update_nxt)
#                     print('output_update_nxt ', State.output_update_nxt)

                # # update modality
                for m in range(update_iters):
                    # measure and don't change resistances
                    State.solve_flow_given_modality(BigClass, "update", access_inters=Sprvsr.access_interNodes)
                    # print('Power', np.sum(State.u**2*State.R_in_t[-1]))

                    State.update_Rs(BigClass)  # supressed - no resistance update
                        
                # # anneal learning rate
                if Sprvsr.anneal and State.t < Sprvsr.iterations:
                    Sprvsr.anneal_alpha(State)
        print('due to update_vec=', State.update_vec)
        print('resulting in resistance=', State.R_in_t[-1])
        # measure accuracy in classification task
        if i % Sprvsr.measure_accuracy_every == 0 and Sprvsr.task_type == 'Iris_classification' \
            and i//Sprvsr.measure_accuracy_every<len(State.accuracy_in_t):
            if i==0:
                State.accuracy_in_t[i] = 1/3
                State.t_for_accuracy[i//Sprvsr.measure_accuracy_every] = State.t 
            else:
                State.calculate_accuracy_testset(BigClass)
                State.accuracy_in_t[i//Sprvsr.measure_accuracy_every] = State.accuracy 
                State.t_for_accuracy[i//Sprvsr.measure_accuracy_every] = State.t 
#     # save R, p, u for network plots, after measurement
#     NET.save_R_reordered(State.R_in_t[-1], Strctr.EIEJ_plots)
#     NET.save_p_reordered(State.p)
#     NET.save_u_reordered(State.u, Strctr.EIEJ_plots)

    # scalar loss in time
    if Sprvsr.task_type == 'Regression':
        denominator_dataset = Sprvsr.y_train
    elif Sprvsr.task_type == 'Iris_classification':
        denominator_dataset = array([[np.mean(State.targets_mat)]])
    if Sprvsr.loss_type == 'MAE':
        State.loss_scalar_in_t = np.mean(np.mean(np.abs(State.loss_in_t), axis=1), axis=1)
        State.loss_scalar_in_t = State.loss_scalar_in_t / np.mean(np.abs(denominator_dataset))
    elif Sprvsr.loss_type == 'MSE':
        State.loss_scalar_in_t = np.mean(np.mean(np.square(State.loss_in_t), axis=1), axis=1)
        State.loss_scalar_in_t = State.loss_scalar_in_t / np.mean(np.square(denominator_dataset))
        # State.loss_scalar_in_t = State.loss_scalar_in_t / np.mean(np.square(denominator_dataset), axis=1)
    
#     if task_type == 'Regression':
#         if loss_type == 'MAE':
#             State.loss_norm_in_t = State.loss_in_t / np.mean(np.abs(Variabs.y_train), axis=1)[:,None,None]
#         elif loss_type == 'MSE':
#             State.loss_norm_in_t = State.loss_in_t / np.mean((Variabs.y_train)**2, axis=1)[:,None,None]
#     elif task_type == 'Iris_classification':
#         if loss_type == 'MAE':
#             State.loss_norm_in_t = State.loss_in_t / np.mean(np.mean(np.abs(Variabs.y_train), axis=1), axis=1)
#         elif loss_type == 'MSE':
#             State.loss_norm_in_t = State.loss_in_t / np.mean(np.mean((Variabs.y_train)**2, axis=1))
#     State.loss_scalar_in_t = np.mean(np.mean(np.abs(State.loss_norm_in_t), axis=1), axis=1) 

In [ ]:
import cProfile
import pstats

if __name__ == "__main__":
    profiler = cProfile.Profile()
    profiler.enable()

    if Sprvsr.task_type == 'Regression':
        for i, alpha in enumerate([CFG.Sprvsr.alpha]):
            print('alpha = ', alpha)
            Sprvsr.assign_alpha(alpha, Variabs)
            State = Network_State(CFG, BigClass)
            # R_mat_i = np.concatenate((1/Sprvsr.M, np.ones([1, 5])), axis=0)
            # R_vec_i = R_mat_i.flatten()
            State.initiate_resistances(BigClass, CFG.State.R_vec_i, add_noise=0.0)
            BigClass.add_State(State)  # add to big class
            training_loop()
            plot_funcs.plot_importants(BigClass, movmean_loss = True, include_network=False, node_labels=False)
            
    elif Sprvsr.task_type == 'Iris_classification':
        accuracy_in_t = []
        random_state = CFG.Sprvsr.random_state
        for i in range(8):
            random_state += 1
            if random_state == 56:
                random_state += 1
            elif random_state == 61:
                random_state += 1
            print('random_state = ', random_state)
            iris_cfg = replace(CFG, Sprvsr=replace(CFG.Sprvsr, random_state=random_state))
            Sprvsr.create_dataset_and_targets(iris_cfg, Strctr, Variabs, train_size=0.2)
            Sprvsr.create_noise_for_extras(Strctr, Variabs)
            State = Network_State(CFG, BigClass)
            State.initiate_resistances(BigClass, CFG.State.R_vec_i)
            State.initiate_accuracy_vec(BigClass)
            BigClass.add_State(State)  # add to big class
            training_loop()
            # plot_funcs.plot_importants(BigClass, movmean_loss = True, include_network=False, node_labels=False)
            accuracy_in_t.append(State.accuracy_in_t)
            # plot_funcs.plot_accuracy(State.t, State.t_for_accuracy, State.accuracy_in_t, np.shape(Variabs.dataset)[0])
    profiler.disable()
    stats = pstats.Stats(profiler).sort_stats('cumtime')  # or 'tottime'

In [ ]:
loss_mean = statistics.final_err(State.loss_scalar_in_t, samples=400)
print('final loss normalized', loss_mean)
print('STD loss normalized', np.std(State.loss_scalar_in_t[-400:]))

loss_movmean = statistics.mov_ave(State.loss_scalar_in_t, 400)
min_mean_loss = loss_movmean[loss_movmean == np.min(loss_movmean)]
print('min_mean_loss', min_mean_loss)

In [ ]:
Sprvsr.M

# Save sizes to file

In [ ]:
save_folder_prelim = 'C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/Network combine/Network_combine/'
# save_folder_prelim = 'C:/Users/roiee/OneDrive - huji.ac.il/PhD/Network Simulation repo/Network combine/Network_combine/'

# # for general regression

# np.save(save_folder_prelim + 't.npy', State.t)
# np.save(save_folder_prelim + 'M.npy', M_values)
# np.save(save_folder_prelim + 'output_lin.npy', np.asarray(State.output_in_t)/np.asarray(State.desired_in_t)-1)
# np.save(save_folder_prelim + 'input_update_lin.npy', State.input_update_in_t)
# np.save(save_folder_prelim + 'output_update_lin.npy', State.output_update_in_t)
# np.save(save_folder_prelim + 'R_lin.npy', State.R_in_t)
# np.save(save_folder_prelim + 'loss_lin.npy', State.loss_in_t)

# import pickle
# # NET.NET networkx graph
# with open(save_folder_prelim + 'NETgraph_4in6out.pkl', 'wb') as f:
#     pickle.dump(NET.NET, f)
    
# # Save the dictionary to a file
# with open(save_folder_prelim + 'pos_lattice_4in6out.pkl', 'wb') as f:
#     pickle.dump(NET.pos_lattice, f)

# # for performance_2_networks
      
# np.save(save_folder_prelim + 'output_4in6out.npy', np.asarray(State.output_in_t))
# np.save(save_folder_prelim + 'input_update_4in6out.npy', State.input_update_in_t)
# np.save(save_folder_prelim + 'output_update_4in6out.npy', State.output_update_in_t)
# np.save(save_folder_prelim + 'R_4in6out_Adalike.npy', State.R_in_t)
# np.save(save_folder_prelim + 'loss_scalar_4in6out_Adalike.npy', State.loss_scalar_in_t)

# import pickle
# # NET.NET networkx graph
# with open(save_folder_prelim + 'NETgraph_1in2out.pkl', 'wb') as f:
#     pickle.dump(NET.NET, f)
    
# # Save the dictionary to a file
# with open(save_folder_prelim + 'pos_lattice_1in2out.pkl', 'wb') as f:
#     pickle.dump(NET.pos_lattice, f)

# np.save(save_folder_prelim + 'cosine_sim_4in6out.npy', cosine_vec)

# for classification_1_material
np.save(save_folder_prelim + 't.npy', State.t)
np.save(save_folder_prelim + 'deltaR_ptopto_Poweraccuracy_in_t.npy', accuracy_in_t)
np.save(save_folder_prelim + 'dataset_shape.npy', np.shape(Sprvsr.dataset))
np.save(save_folder_prelim + 't_for_accuracy.npy', State.t_for_accuracy)

# for classification_5_material
# np.save(save_folder_prelim + 't.npy', State.t)
# np.save(save_folder_prelim + 'deltaR_propto_dpaccuracy_in_t.npy', accuracy_in_t)
# np.save(save_folder_prelim + 'dataset_shape.npy', np.shape(Variabs.dataset))
# np.save(save_folder_prelim + 't_for_accuracy.npy', State.t_for_accuracy)

# Plots - specific run

## Structure

In [ ]:
# plot_funcs.plotNetStructure(NET.NET, BigClass, NET.pos_lattice, node_labels=False, 
#                                 R_reordered=NET.R_reordered, u_reordered=NET.u_reordered, p_reordered=NET.p_reordered)

## ||Loss|| after movmean

In [ ]:
loss_scalar_moveave = statistics.mov_ave(State.loss_scalar_in_t, 2000)

plt.rcParams['axes.prop_cycle'] = plt.cycler('color', colors_lst)
plt.semilogy(loss_scalar_moveave)
plt.ylabel('$\|\mathcal{L}\|$')
plt.xlabel('$t$')
plt.ylim([2e-3,1]);

## accuracy

In [ ]:
if Sprvsr.task_type == 'Iris_classification':
    plot_funcs.plot_accuracy(State.t, State.t_for_accuracy, State.accuracy_in_t, np.shape(Sprvsr.dataset)[0])
else:
    pass

# Not In Use

## Power consumption

In [ ]:
# if np.size(State.Power_norm_in_t)>1:
#     plot_funcs.plot_Power(State)
# else:
#     pass

## Power of trained network

put a pressure of 1 through all inputs and measure total power dissipation in a trained network that has the state State

In [ ]:
# # Reload the module to reflect any changes made
# importlib.reload(statistics)

# # put pressure of 1 through inputs
# State.input_drawn = np.ones(Nin)

# # solve flow
# State.solve_flow_given_modality(BigClass, "measure")

# # measure power
# print('u', State.u)
# print('Rs', State.R_in_t[-1])
# print('Power dissipation', statistics.power_dissip(State.u, State.R_in_t[-1]))

### R change scheme under 2 tasks

In [ ]:
# import pickle

# load_folder_prelim = 'C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.9/importants_1in2out_n_2in1out/'
# # load_folder_prelim = 'C:/Users/roiee/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.9/importants_1in2out_n_2in1out/'
# t = np.load(load_folder_prelim + 't_1in2out.npy')

# loss_1in2out_R_propto_deltap = np.load(load_folder_prelim + 'loss_1in2out_RproptoDeltap.npy')
# loss_1in2out_deltaR_propto_deltap = np.load(load_folder_prelim + 'loss_1in2out_deltaRproptoDeltap.npy')
# loss_1in2out_propto_Q = np.load(load_folder_prelim + 'loss_1in2out_deltaRproptoQ.npy')
# loss_1in2out_propto_Power = np.load(load_folder_prelim + 'loss_1in2out_deltaRproptoPower.npy')
# loss_2in1out_R_propto_deltap = np.load(load_folder_prelim + 'loss_2in1out_RproptoDeltap.npy')
# loss_2in1out_deltaR_propto_deltap = np.load(load_folder_prelim + 'loss_2in1out_deltaRproptoDeltap.npy')
# loss_2in1out_propto_Q = np.load(load_folder_prelim + 'loss_2in1out_deltaRproptoQ.npy')
# loss_2in1out_propto_Power = np.load(load_folder_prelim + 'loss_2in1out_deltaRproptoPower.npy')

# with open(load_folder_prelim + 'NETgraph_1in2out.pkl', 'rb') as f:
#     Network_1in2out = pickle.load(f)
    
# with open(load_folder_prelim + 'NETgraph_2in1out.pkl', 'rb') as f:
#     Network_2in1out = pickle.load(f)
    
# with open(load_folder_prelim + 'pos_lattice_2in1out.pkl', 'rb') as f:
#     pos_lattice = pickle.load(f) 

In [ ]:
# # Reload the module to reflect any changes made
# importlib.reload(figure_plots)

# figure_plots.plot_compare_R_type_loss(Network_1in2out, Network_2in1out, pos_lattice,
#                          loss_1in2out_R_propto_deltap,
#                          loss_1in2out_deltaR_propto_deltap,
#                          loss_1in2out_propto_Q,
#                          loss_1in2out_propto_Power,
#                          loss_2in1out_R_propto_deltap,
#                          loss_2in1out_deltaR_propto_deltap,
#                          loss_2in1out_propto_Q,
#                          loss_2in1out_propto_Power)

## different R relations

In [ ]:
# load_folder_prelim = 'C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.9/different physical dependence of resistance/'

# # Loading the array later


# R_propto_deltap = np.load(load_folder_prelim + 'R_propto_deltap.npy')
# deltaR_propto_deltap = np.load(load_folder_prelim + '/deltaR_propto_deltap.npy')
# deltaR_propto_Q = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.9/different physical dependence of resistance/deltaR_propto_Q.npy')
# deltaR_propto_Power = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.9/different physical dependence of resistance/deltaR_propto_Power.npy')
# loss_R_propto_deltap = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.9/different physical dependence of resistance/loss_R_propto_deltap.npy')
# loss_deltaR_propto_deltap = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.9/different physical dependence of resistance/loss_deltaR_propto_deltap.npy')
# loss_propto_Q = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.9/different physical dependence of resistance/loss_propto_Q.npy')
# loss_propto_Power = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.9/different physical dependence of resistance/loss_propto_Power.npy')

In [ ]:
# figure_plots.plot_comparison_R_type(R_propto_deltap, deltaR_propto_deltap, deltaR_propto_Q, deltaR_propto_Power,
#                                       loss_R_propto_deltap, loss_deltaR_propto_deltap, loss_propto_Q, loss_propto_Power)

## pseudo vs. network comparison

In [ ]:
# # Loading the array later
# R_in_t_network = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.8/30Aug_R_pseudo_vs_network/R_in_t_network.npy')
# R_in_t_pseudo = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.8/30Aug_R_pseudo_vs_network/R_in_t_pseudo.npy')
# loss_in_t_network = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.8/30Aug_R_pseudo_vs_network/loss_in_t_network.npy')
# loss_in_t_pseudo = np.load('C:/Users/SMR_Admin/OneDrive - huji.ac.il/PhD/Network Simulation repo/figs and data/2024.8/30Aug_R_pseudo_vs_network/loss_in_t_pseudo.npy')

In [ ]:
# figure_plots.plot_comparison_pseudo(R_in_t_pseudo, R_in_t_network, loss_in_t_pseudo, loss_in_t_network)

## comparison Rs GD and Adalike

In [ ]:
# import matplotlib.gridspec as gridspec

# # Set color cycle globally
# plt.rcParams['axes.prop_cycle'] = plt.cycler('color', Colorscheme.colors_lst)

# # Normalize R values so maximal will be 1
# R_GD_norm = R_GD / np.max(R_GD)
# R_ourscheme_norm = R_ourscheme / np.max(R_ourscheme)

# # R_GD_norm = np.concatenate([R_GD_norm[:-1], R_GD_norm[-1:]])
# # R_ourscheme_norm = np.concatenate([R_ourscheme_norm[:-1],  R_ourscheme_norm[-1:]])

# # weird setup for bars
# x = np.arange(len(R_ourscheme_norm))
# bar_width = 0.35

# # instantiate figure and grid for positioning colorbal
# fig = plt.figure(figsize=(7, 3))
# gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1], wspace=0.45)

# # R bar plot
# ax0 = fig.add_subplot(gs[0, 0])
# ax0.bar(x - bar_width / 2, R_GD_norm,
#         width=bar_width, label='GD', alpha=0.8, edgecolor='k', linewidth=1.6)
# ax0.bar(x + bar_width / 2, R_ourscheme_norm,
#         width=bar_width, label='this work', alpha=0.8, edgecolor='k', linewidth=1.6)
# ax0.set_xticks([0, 1, 2, 3])
# ax0.set_yticks([0, 0.5, 1])
# ax0.set_ylabel('$R$')
# # ax0.legend()

## Is the network linear?

In [ ]:
# # put pressure of 1 through 1st input
# State.input_drawn = np.array([1,0])

# # solve flow
# State.solve_flow_given_modality(BigClass, "measure")

# # measure power
# print('output 1', State.output)
# out1 = State.output

# # put pressure of 1 through 1st input
# State.input_drawn = np.array([0,1])

# # solve flow
# State.solve_flow_given_modality(BigClass, "measure")

# # measure power
# print('output 2', State.output)
# out2 = State.output

# print('superpose outputs', out1+out2)

# # put pressure of 1 through inputs
# State.input_drawn = np.ones(Nin)

# # solve flow
# State.solve_flow_given_modality(BigClass, "measure")

# # measure power
# print('output both', State.output)

## Most important plot

In [ ]:
importlib.reload(plot_funcs)
if hasattr(Variabs, 'M'):
    plot_funcs.plot_importants(BigClass, movmean_loss = True, include_network=False, node_labels=False)
else:
    plot_funcs.plot_importants(BigClass, include_network=False)

In [ ]:
Strctr.EIEJ_plots[:-6]

## cosine similarity

In [ ]:
if Sprvsr.calculate_cosine_sim:
    plt.rcParams['axes.prop_cycle'] = plt.cycler('color', colors_lst)

    loss_movmean = statistics.mov_ave(State.loss_scalar_in_t, 40)

    # Create subplots: 2 rows, 1 column
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, sharex=True, figsize=(6, 5))

    # First subplot: resistances
    ax1.plot(BigClass.State.R_in_t[1:])
    ax1.set_ylabel(r'$R$')

    # Second subplot: Cosine similarity
    ax2.set_prop_cycle(plt.cycler('color', colors_lst))  # set color cycle
    ax2.plot(cosine_vec)
    ax2.plot(statistics.mov_ave(cosine_vec, window_size=24))
    ax2.plot(np.zeros([len(cosine_vec)]), '--k')
    ax2.set_ylabel('Cos Similarity')
    ax2.set_ylim([-1, 1])

    # Third subplot: Loss moving average
    ax3.plot(loss_movmean)
    ax3.set_xlabel('$t$')
    ax3.set_ylabel(r'$\|\mathcal{L}\|$')
    ax3.set_yscale('log')
    ax3.set_ylim(1e-4, 1e0)

    plt.tight_layout()
    plt.show()
    
    plt.plot(cosine_vec)
    plt.plot(np.zeros([len(cosine_vec)]), '--k')
    plt.ylabel('Cos Similarity')
    plt.ylim([-1, 1])
    plt.xlabel('$t$')

In [ ]:
np.nanmean(cosine_vec)

In [ ]:
np.mean(cosine_vec[np.isnan(cosine_vec)==False])